# Drum-Audio-Separator-U_Net
**Autor:** Philipe Amâncio Reis Caetano


## 1. Objetivo do projeto

Desenvolver uma U-Net em PyTorch para estimar uma máscara tempo-frequência capaz de separar a bateria (`drums`) de uma mistura musical.

### Entrada
- `mixture.wav`
- Áudio mono a 44.100 Hz
- Magnitude do espectrograma STFT
- Tensor no formato `[batch, 1, 1024, 512]`

### Saída
- Máscara espectral estimada entre 0 e 1
- Espectrograma estimado da bateria
- Áudio reconstruído `drums_predicted.wav`

## 2. Configurações

### 2.1. Configurações do ambiente

Inicialmente, foram importadas as bibliotecas necessárias para manipulação de arquivos, processamento digital de áudio, operações numéricas e implementação do modelo de aprendizado profundo. A biblioteca Librosa foi utilizada para carregamento dos arquivos WAV e cálculo da Transformada de Fourier de Curto Tempo (STFT), enquanto NumPy foi empregada na manipulação dos espectrogramas e máscaras espectrais. O framework PyTorch foi utilizado para construir e treinar a arquitetura U-Net, e seus componentes Dataset e DataLoader foram usados para organizar os exemplos de áudio em batches. Por fim, foi realizada uma verificação da disponibilidade de CUDA, confirmando a utilização da GPU NVIDIA GeForce GTX 1650 para acelerar as operações de treinamento.

In [5]:
from pathlib import Path
import random
import gc
import time

import numpy as np
import librosa
import soundfile as sf

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim

from torch.utils.data import Dataset, DataLoader

print("PyTorch:", torch.__version__)
print("CUDA disponível:", torch.cuda.is_available())

if torch.cuda.is_available():
    print(
        "GPU:",
        torch.cuda.get_device_name(0)
    )

PyTorch: 2.5.1+cu121
CUDA disponível: True
GPU: NVIDIA GeForce GTX 1650


### 2.2. Configurações locais

Em seguida, foram definidos os parâmetros globais do experimento. Uma semente aleatória de valor 42 foi aplicada às bibliotecas Python, NumPy e PyTorch, incluindo CUDA, com o objetivo de reduzir variações entre execuções e favorecer a reprodutibilidade dos resultados. O dispositivo de processamento foi configurado automaticamente para utilizar CUDA quando uma GPU NVIDIA compatível estivesse disponível, sendo utilizada a GPU GeForce GTX 1650.

Também foram definidos os caminhos locais do dataset MUSDB18-HQ e da pasta de checkpoints, utilizada para armazenar os estados do modelo ao longo do treinamento. Para a representação tempo-frequência, foi adotada **taxa de amostragem de 44.100 Hz**, tamanho de **janela FFT de 2.048 amostras** e **salto de 512 amostras**. Essa configuração produz **1.024 bins de frequência** utilizados como entrada e gera frames temporais a cada aproximadamente **11,6 ms**, com **75% de sobreposição entre janelas**. Cada exemplo foi dividido em segmentos de **256 frames**, correspondentes a aproximadamente **3 segundos de áudio**. Por fim, foi definido batch size igual a 1 e carregamento sem workers paralelos, priorizando estabilidade e compatibilidade com a memória disponível na GPU.

In [6]:
SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)

DATASET_ROOT = Path.cwd() / "musdb18hq" #Path.cwd() retorna o o diretório atual
CHECKPOINT_DIR = Path.cwd() / "checkpoints"

CHECKPOINT_DIR.mkdir( #Cria a pasta caso não exista
    parents=True,
    exist_ok=True
)

SAMPLE_RATE = 44100
N_FFT = 2048
HOP_LENGTH = 512

FRAMES_PER_CHUNK = 256
BATCH_SIZE = 1
NUM_WORKERS = 0

print("Dispositivo:", device)
print("Dataset:", DATASET_ROOT)
print("Dataset existe:", DATASET_ROOT.exists())
print("Checkpoints:", CHECKPOINT_DIR)

Dispositivo: cuda
Dataset: C:\Users\phili\Documents\IA-II\musdb18hq
Dataset existe: True
Checkpoints: C:\Users\phili\Documents\IA-II\checkpoints


* **Sample Rate de 44100 Hz**
Um ouvido humano saudável consegue captar frequências de 20 Hz a 20.000 Hz. Pelo Teorema de Nyquist, precisamos do dobro dessa frequência (40.000 Hz) para conseguirmos registrar perfeitamente os pontos de topo e vale de cada onda. Na prática, usamos **44.100 Hz** em vez de 40.000 Hz cravados. Esses 4.100 Hz extras servem como uma "zona de respiro" (margem de segurança) para os filtros atuarem suavemente, evitando um defeito digital chamado Aliasing.

  * **Aliasing:** É uma distorção severa que ocorre quando uma frequência ultrapassa o limite máximo que o computador consegue registrar (o limite de Nyquist). Como a quantidade ciclos do computador não é o suficiente para capturar os topos e vales desse som muito agudo, ela acaba ligando os pontos de forma errada. O resultado é que esse som não é simplesmente ignorado; ele "dobra" e vaza de volta para a região audível como uma frequência fantasma, grave e metálica, sujando a gravação. É o equivalente sonoro daquela ilusão de ótica em vídeos onde a roda de um carro girando muito rápido parece estar girando para trás.

* **Hop Length**
Define a resolução temporal da nossa imagem. É o "passo", ou seja, a quantidade de amostras de áudio que a janela de análise avança no eixo do tempo antes de calcular a próxima Transformada de Fourier.

  * **A Matemática:** Em um áudio a 44.100 Hz, um salto de 512 amostras equivale a um avanço de tempo de: $T_{hop} = \frac{512}{44100} \approx 11.6 \text{ milissegundos}$. Isso significa: "Se passou 512 samples e está tocando 44.1000 samples em 1 segundo, quanto da muúsica tocou?"

  * **A Sobreposição (Overlap):** Como a nossa janela total de análise tem 2048 amostras, mas avançamos apenas 512 por vez, estamos sobrepondo 75% da informação da "foto" anterior na nova "foto".

  * **O Motivo (Preservação de Transientes):** No mundo da guitarra, a palhetada (o ataque do *palm mute*, por exemplo) é um som explosivo, caótico e muito rápido, chamado de **transiente**. Se a janela de análise pulasse 2048 amostras de uma vez (sem sobreposição), esse impacto rápido poderia cair exatamente na emenda entre duas janelas e ser borrado ou perdido. O salto curto de 512 garante que a "câmera" do STFT capture o momento exato do impacto da palheta em múltiplas fotos sobrepostas. Visualmente, isso cria linhas verticais extremamente nítidas no espectrograma, permitindo que a U-Net identifique não apenas a afinação da guitarra, mas a marcação rítmica perfeita da música.

  * **Por que exatamente 512?**
    
    A escolha do valor 512 não é arbitrária, mas sim a convergência de três regras fundamentais de processamento de sinais e computação:

    1. **Otimização de Hardware (Potências de 2):** O algoritmo matemático que calcula as frequências (*Fast Fourier Transform* - FFT) exige que os blocos de dados sejam potências de 2 ($2^n$) para rodar com eficiência máxima na GPU. Um valor como "500" quebraria a otimização binária, tornando o processamento drasticamente mais lento.
    2. **A Fração de 1/4 (75% de Sobreposição):** Na reconstrução de sinais de áudio, a proporção ideal para evitar perda de dados é avançar a janela em exatamente $1/4$ do seu tamanho total. Como nosso `n_fft` é de 2048 amostras, o salto ideal é $\frac{2048}{4} = 512$.
    3. **A Suavização (Janela de Hann):** Para evitar "estalos" digitais nas bordas de cada recorte de áudio, aplica-se uma máscara matemática que faz um *fade in / fade out* nas pontas da janela (Janelamento). Com um salto exato de 512 amostras, o *fade out* de um recorte se soma com perfeição ao *fade in* do próximo. A matemática se anula e o volume da música se mantém contínuo, sem pulsações indesejadas.


---

## 3. Verificação e organização do diretório do dataset

O dataset MUSDB18-HQ foi organizado em dois subconjuntos: treino e teste. Foram identificadas `100 faixas` no diretório de treino e `50 faixas` no diretório de teste. Cada faixa é armazenada em uma subpasta individual, contendo a mistura musical completa (mixture.wav) e os stems isolados dos instrumentos. Para o problema de separação de bateria, foram utilizados principalmente o arquivo mixture.wav, que representa a entrada da rede, e o arquivo drums.wav, que representa a referência supervisionada. A divisão entre treino e teste permite ajustar os parâmetros da U-Net somente com músicas do conjunto de treinamento e avaliar sua capacidade de generalização em faixas não utilizadas na atualização dos pesos.

In [7]:
# Define os caminho do diretório do dataset
train_dir = DATASET_ROOT / "train"
test_dir = DATASET_ROOT / "test"

# Organiza o diretório
train_tracks = sorted(
    folder
    for folder in train_dir.iterdir()
    if folder.is_dir()
)

test_tracks = sorted(
    folder
    for folder in test_dir.iterdir()
    if folder.is_dir()
)

print("Faixas de treino:", len(train_tracks))
print("Faixas de teste:", len(test_tracks))

first_track = train_tracks[0] # Seleciona a primeira track para teste do pipeline

print("Primeira faixa:", first_track.name)
print(
    "mixture.wav:",
    (first_track / "mixture.wav").exists()
)
print(
    "drums.wav:",
    (first_track / "drums.wav").exists()
)

Faixas de treino: 100
Faixas de teste: 50
Primeira faixa: A Classic Education - NightOwl
mixture.wav: True
drums.wav: True


---

## [TESTE_PIPELINE]. Validação do Carregamento de Áudio

Antes de processar todo o dataset, esta etapa carrega um trecho inicial de uma única música para validar a matemática do recorte e garantir que o pipeline de áudio está funcionando corretamente.

**Lógica do Recorte:**
* **Cálculo de Amostras:** Para que a rede neural receba exatamente 256 frames temporais no espectrograma, é necessário carregar a quantidade correta de áudio bruto. A fórmula `(FRAMES_PER_CHUNK * HOP_LENGTH) + N_FFT` garante o tamanho exato do trecho necessário para que a janela da STFT deslize até o último frame sem perder dados.
* **Conversão para Segundos:** A função `librosa.load` exige os parâmetros de início (`offset`) e duração (`duration`) em segundos. Para converter nossa janela de análise para tempo físico, o total de amostras digitais é dividido pela taxa de amostragem (`SAMPLE_RATE`).
* **Sincronia:** O carregamento espelhado de `mixture.wav` (entrada) e `drums.wav` (alvo supervisionado) assegura que ambos os tensores possuam o mesmo comprimento (133.120 amostras para ~3 segundos) e a mesma taxa de amostragem (44.100 Hz).

In [8]:
track = train_tracks[0] # Pega a primeira track para validar o pipeline


sample_start = 0 # Marca o início do trecho

# Matém o padrão de tamnho do audio, caso a janela vaze para além do tamanho da faixa de audio em um hop
audio_length = (
    FRAMES_PER_CHUNK * HOP_LENGTH
    + N_FFT
)

# Carrega o trecho da mistura (mixture.wav)
y_mix, sr_mix = librosa.load(
    track / "mixture.wav",
    sr=SAMPLE_RATE,
    mono=True,
    offset=sample_start / SAMPLE_RATE,
    duration=audio_length / SAMPLE_RATE
)


# Carrega o trecho da bateria isolada (drum.wav)
y_drums, sr_drums = librosa.load(
    track / "drums.wav",
    sr=SAMPLE_RATE,
    mono=True,
    offset=sample_start / SAMPLE_RATE,
    duration=audio_length / SAMPLE_RATE
)

print("Amostras mixture:", len(y_mix))
print("Amostras drums:", len(y_drums))
print("Sample rate:", sr_mix, sr_drums)
print(
    "Duração do trecho:",
    round(len(y_mix) / SAMPLE_RATE, 2),
    "segundos"
) 

Amostras mixture: 133120
Amostras drums: 133120
Sample rate: 44100 44100
Duração do trecho: 3.02 segundos


### [TESTE_PIPELINE] Extração da STFT e Geração da Máscara-Alvo

Esta etapa disseca a transformação matemática central do projeto. Aqui, o áudio bruto é convertido em uma representação visual (espectrograma) e a "resposta certa" (máscara-alvo) é gerada para que a U-Net possa aprender.

**Passo a passo da transformação:**
* **STFT e Magnitude:** A Transformada de Fourier de Curto Tempo converte a onda sonora em uma matriz 2D de números complexos (frequência × tempo). Utilizamos `np.abs` para isolar apenas a magnitude (volume/energia), descartando temporariamente a fase.
* **Ajuste de Dimensão (`[:-1]`):** A STFT padrão retorna 1025 bins de frequência. Removemos o último bin (limite de Nyquist, que possui pouca informação útil) para obtermos **1024 bins**. Esse número exato é obrigatório para a arquitetura U-Net, pois é perfeitamente divisível por 16, permitindo que a imagem passe pelas camadas de *Pooling* sem quebrar dimensões.
* **A Máscara-Alvo (Target Mask):** O objetivo da rede é estimar qual fração da música é bateria. Calculamos isso dividindo a magnitude da bateria pela mistura (`drums_mag / mix_mag`). O fator `1e-8` é adicionado ao denominador como uma trava de segurança contra divisão por zero em momentos de silêncio absoluto.
* **Conversão Logarítmica e Normalização:** O ouvido humano percebe o volume de forma não-linear. O uso de `librosa.amplitude_to_db` comprime a gigantesca variação de picos da magnitude para uma escala estabilizada de Decibéis (onde o pico é 0 e o silêncio é -80). Em seguida, essa escala é esmagada para ficar entre **0.0 e 1.0**, formato numérico ideal para evitar a explosão de gradientes durante o treinamento da rede neural.

In [9]:
# Calcula o STFT da mistura musical
# Converto onda sonora em espectograma
mix_stft = librosa.stft(
    y_mix,
    n_fft=N_FFT, # 
    hop_length=HOP_LENGTH
)

# Calcula o STFT da bateria isolada
drums_stft = librosa.stft(
    y_drums,
    n_fft=N_FFT,
    hop_length=HOP_LENGTH
)


mix_mag = np.abs(mix_stft)[:-1] # Calcula a magnitude dos nums complexos da matriz mix_stf (O quão ativado está o bin) [Entrada}
drums_mag = np.abs(drums_stft)[:-1] # Calcula a magnitude dos nums complexos da matriz drums_stft  [Alvo]

# Máscara ideal
target_mask = drums_mag / (mix_mag + 1e-8) ## Sobra a matriz com valores de 0 a 1, quanto mais próximo de 1 mais bateria tem
target_mask = np.clip(target_mask, 0, 1)

# Transforma em magnitude logarítimica, evitando explosões de gradientes no treino
mix_db = librosa.amplitude_to_db(
    mix_mag,
    ref=np.max
)

# Padroniza os valores da matriz_db para [0,1], sendo 0 silêncio e 1 som ativo.
mix_input = np.clip(
    (mix_db + 80.0) / 80.0,
    0.0,
    1.0
)

print("STFT mixture:", mix_stft.shape)
print("Entrada:", mix_input.shape)
print("Máscara alvo:", target_mask.shape)
print(
    "Faixa da entrada:",
    round(float(mix_input.min()), 4),
    "até",
    round(float(mix_input.max()), 4)
)
print(
    "Faixa da máscara:",
    round(float(target_mask.min()), 4),
    "até",
    round(float(target_mask.max()), 4)
)

STFT mixture: (1025, 261)
Entrada: (1024, 261)
Máscara alvo: (1024, 261)
Faixa da entrada: 0.0 até 1.0
Faixa da máscara: 0.0001 até 1.0


### [TESTE_PIPELINE] Formatação de Tensores (Batch, Channel, Freq, Time)

Para que o espectrograma seja processado pelas camadas convolucionais da U-Net no PyTorch, as matrizes bidimensionais do NumPy precisam ser convertidas em tensores 4D estritos. 

**Operações de Empacotamento:**
* **Recorte Temporal (Slicing):** O espectrograma contínuo é fatiado para conter um número fixo de frames (`FRAMES_PER_CHUNK = 256`), garantindo que a rede neural receba imagens com larguras idênticas (~3 segundos de áudio).
* **Máscara de Validação (Valid Mask):** Uma matriz de sinalização é gerada. Durante o treinamento real, blocos no final da música podem ser preenchidos artificialmente com zeros (padding) para completar 256 frames. Esta máscara sinaliza à função de Loss (Erro) quais frames são áudio real para que o modelo não seja penalizado por errar predições no silêncio artificial.
* **Injeção de Dimensões (`unsqueeze`):** O PyTorch exige o formato de entrada `[Lote, Canais, Altura, Largura]`. Como o nosso espectrograma bidimensional possui apenas Altura (1024 bins) e Largura (256 frames), utilizamos o `.unsqueeze(0)` duas vezes para adicionar a dimensão de Lote (`BATCH_SIZE = 1`) e a dimensão de Canais (1 canal acústico, simulando uma imagem monocromática).

In [10]:
mix_chunk = mix_input[
    :, :FRAMES_PER_CHUNK
]

mask_chunk = target_mask[
    :, :FRAMES_PER_CHUNK
]

valid_mask = np.ones(
    (1, FRAMES_PER_CHUNK),
    dtype=np.float32
)

mixture_tensor = torch.tensor(
    mix_chunk,
    dtype=torch.float32
).unsqueeze(0).unsqueeze(0)

target_tensor = torch.tensor(
    mask_chunk,
    dtype=torch.float32
).unsqueeze(0).unsqueeze(0)

valid_mask_tensor = torch.tensor(
    valid_mask,
    dtype=torch.float32
).unsqueeze(0)

print("Mixture:", mixture_tensor.shape)
print("Target:", target_tensor.shape)
print("Valid mask:", valid_mask_tensor.shape)

Mixture: torch.Size([1, 1, 1024, 256])
Target: torch.Size([1, 1, 1024, 256])
Valid mask: torch.Size([1, 1, 256])


---

## 4. Arquitetura U-Net

### 4.1. ConvBlock (Extração de Características)

O bloco fundamental de operação da arquitetura. Ele aplica duas camadas convolucionais sequenciais (`Conv2d`) para extrair padrões acústicos da matriz (como o impacto de um bumbo ou a frequência contínua de um prato). Cada convolução é seguida por um `InstanceNorm2d`, que estabiliza matematicamente o contraste das frequências para a amostra atual (indispensável para treinamentos com batch size unitário), e uma função de ativação não-linear `ReLU`.

In [11]:
class ConvBlock(nn.Module):
    def __init__(
        self,
        in_channels,
        out_channels
    ):
        super().__init__()

        self.net = nn.Sequential(
            nn.Conv2d(
                in_channels,
                out_channels,
                kernel_size=3,
                padding=1
            ),
            nn.InstanceNorm2d(out_channels),
            nn.ReLU(inplace=True),

            nn.Conv2d(
                out_channels,
                out_channels,
                kernel_size=3,
                padding=1
            ),
            nn.InstanceNorm2d(out_channels),
            nn.ReLU(inplace=True)
        )

    def forward(self, x):
        return self.net(x)



**Nota de Arquitetura: Por que InstanceNorm2d em vez de BatchNorm2d?**

Em arquiteturas Convolutionais tradicionais, o `BatchNorm2d` é o padrão para estabilizar gradientes. No entanto, ele calcula a média e a variância com base em múltiplos exemplos de um lote, construindo um histórico (média móvel) para usar durante a inferência. 

Devido à limitação de VRAM, este projeto utiliza `BATCH_SIZE = 1`. Com um lote unitário, as estatísticas históricas do `BatchNorm` tornam-se altamente erráticas, pois cada passo de treino representa um extremo acústico isolado (ex: um pico de bumbo vs. silêncio). Durante o `model.eval()`, a aplicação desse histórico ruidoso destrói as dinâmicas do espectrograma, gerando artefatos destrutivos (SI-SDR negativo).

A solução empregada é o **`InstanceNorm2d`**. Ele calcula a normalização espacial de forma independente para cada exemplo (instância) e canal, sem depender de estatísticas globais do dataset. Isso preserva as variações locais de amplitude do áudio e garante total estabilidade matemática mesmo operando com um lote de tamanho 1.

### 4.2. EncoderBlock (Downsampling e Memória)

Responsável pela etapa de codificação e compressão. O espectrograma é processado pelo `ConvBlock` e, em seguida, sua resolução espacial é reduzida pela metade utilizando `MaxPool2d`. Essa compressão permite que a rede amplie seu campo de visão para entender contextos rítmicos maiores. Crucialmente, antes de comprimir a matriz, o bloco salva uma cópia intacta da mesma (o `skip connection`) para preservar a precisão temporal em alta resolução para o futuro.

In [12]:
class EncoderBlock(nn.Module):
    def __init__(
        self,
        in_channels,
        out_channels
    ):
        super().__init__()

        self.conv = ConvBlock(
            in_channels,
            out_channels
        )

        self.pool = nn.MaxPool2d(
            kernel_size=2,
            stride=2
        )

    def forward(self, x):
        skip = self.conv(x)
        pooled = self.pool(skip)

        return pooled, skip


### 4.3. BottleneckBlock (Ponto de Abstração Máxima)

Representa o ponto mais profundo da arquitetura em formato de "U". Neste estágio, a resolução temporal e de frequência atingiu sua compressão máxima, enquanto o número de canais de extração (filtros) é o mais alto. O modelo foca integralmente na representação abstrata e complexa dos timbres presentes na mistura, atuando como a ponte entre a codificação e a reconstrução.

In [13]:
class BottleneckBlock(nn.Module):
    def __init__(
        self,
        in_channels,
        out_channels
    ):
        super().__init__()

        self.conv = ConvBlock(
            in_channels,
            out_channels
        )

    def forward(self, x):
        return self.conv(x)


### 4.4. DecoderBlock (Upsampling e Reconstrução Espacial)

Responsável por realizar o caminho inverso de decodificação. O bloco expande a matriz comprimida (`nn.Upsample` com interpolação bilinear) de volta ao dobro do tamanho. O passo mais importante ocorre com a função `torch.cat`, que concatena a imagem em expansão com o seu `skip connection` correspondente, importado diretamente do Encoder. Essa fusão devolve à rede a precisão espacial necessária para desenhar a máscara final exatamente sobre os transientes da bateria, sem borrar o tempo.

In [14]:
class DecoderBlock(nn.Module):
    def __init__(
        self,
        in_channels,
        out_channels
    ):
        super().__init__()

        self.up = nn.Upsample(
            scale_factor=2,
            mode="bilinear",
            align_corners=False
        )

        self.conv = ConvBlock(
            in_channels,
            out_channels
        )

    def forward(self, x, skip):
        x = self.up(x)

        if x.shape[-2:] != skip.shape[-2:]:
            x = F.interpolate(
                x,
                size=skip.shape[-2:],
                mode="bilinear",
                align_corners=False
            )

        x = torch.cat(
            [x, skip],
            dim=1
        )

        return self.conv(x)

---

## 5. Arquitetura Principal (`UNetAudio`)

Esta classe orquestra a montagem completa da U-Net, conectando o codificador, o centro de abstração e o decodificador em um fluxo contínuo de ponta a ponta.

**Estratégia Estrutural:**
* **Expansão de Canais:** A rede dobra progressivamente a quantidade de filtros convolucionais no Encoder (`1` $\to$ `16` $\to$ `32` $\to$ `64` $\to$ `128` $\to$ `256`), permitindo que camadas mais profundas capturem padrões harmônicos e rítmicos cada vez mais complexos.
* **O Fluxo de Encaminhamento (`forward`):** O espectrograma de entrada passa sequencialmente pelos blocos de codificação (`enc1` a `enc4`), salvando os respectivos `skips` para preservar a precisão temporal. No ponto mais profundo (`bottleneck`), a informação é processada em máxima abstração e devolvida subindo pelo Decoder (`dec1` a `dec4`), onde é fundida com os desvios salvos.
* **Camada de Saída e Ativação:** Uma convolução final de kernel 1x1 reduz os canais internos de volta a 1. Em seguida, a função **`torch.sigmoid`** restringe matematicamente todos os valores da matriz de saída para o intervalo estrito de `0.0` a `1.0`. **Essa matriz resultante funciona como uma "máscara de probabilidade": valores próximos de `1.0` indicam que o modelo tem alta certeza de que aquela frequência/momento pertence à bateria, enquanto valores próximos de `0.0` representam silêncio ou outros instrumentos.** Isso garante que a predição da rede corresponda exatamente à escala da máscara-alvo usada no treino.

In [15]:
class UNetAudio(nn.Module):
    def __init__(self):
        super().__init__()

        self.enc1 = EncoderBlock(1, 16)      
        self.enc2 = EncoderBlock(16, 32)     
        self.enc3 = EncoderBlock(32, 64)     
        self.enc4 = EncoderBlock(64, 128)    

        # BOTTLENECK: Centro da rede, onde a abstração é mais profunda
        self.bottleneck = BottleneckBlock(
            128,
            256                              
        )

        # DECODER: Reconstruindo a imagem com as conexões de pulo (skip connections)
        self.dec1 = DecoderBlock(
            256 + 128,                       
            128
        )

        self.dec2 = DecoderBlock(
            128 + 64,                        
            64
        )

        self.dec3 = DecoderBlock(
            64 + 32,                         
            32
        )

        self.dec4 = DecoderBlock(
            32 + 16,                         
            16
        )

        # FINAL: Reduzindo de volta para 1 canal (a máscara espectral)
        self.final_conv = nn.Conv2d(
            16,
            1,
            kernel_size=1
        )

    def forward(self, x):
        x, skip1 = self.enc1(x)
        x, skip2 = self.enc2(x)
        x, skip3 = self.enc3(x)
        x, skip4 = self.enc4(x)

        x = self.bottleneck(x)

        x = self.dec1(x, skip4)
        x = self.dec2(x, skip3)
        x = self.dec3(x, skip2)
        x = self.dec4(x, skip1)

        return torch.sigmoid(
            self.final_conv(x)
        )

---

### [TESTE_PIPELINE] Smoke Test (Teste de Fumaça) da U-Net na GPU

Esta etapa executa um teste preliminar de encaminhamento (*forward pass*) enviando um bloco de áudio fictício diretamente para a placa de vídeo. 

**Objetivos do Teste:**
* **Validação de Formato:** Assegura que o tensor de entrada (`[1, 1, 1024, 256]`) transita sem erros de dimensão por todas as camadas de codificação e decodificação, retornando uma saída com o formato idêntico (`[1, 1, 1024, 256]`).
* **Modo de Inferência (`torch.no_grad()`):** Desativa o motor de cálculo de gradientes, simulando o comportamento de uso real e isolando o custo computacional puro.
* **Auditoria de Desempenho e VRAM:** Utiliza ferramentas de sincronização da GPU (`torch.cuda.synchronize`) e rastreamento de pico (`max_memory_allocated`) para mensurar com precisão o tempo de processamento e o consumo exato de memória de vídeo, garantindo a viabilidade de execução na GPU local.

In [16]:
model_unet = UNetAudio().to(device)

mixture_gpu = mixture_tensor.to(device)

torch.cuda.reset_peak_memory_stats()

start = time.time()

with torch.no_grad():
    output = model_unet(mixture_gpu)

torch.cuda.synchronize()

print("Entrada:", mixture_gpu.shape)
print("Saída:", output.shape)
print(
    "Tempo:",
    round(time.time() - start, 3),
    "segundos"
)
print(
    "VRAM máxima:",
    round(
        torch.cuda.max_memory_allocated()
        / 1024**3,
        2
    ),
    "GB"
)
print(
    "Máscara:",
    round(float(output.min()), 4),
    "até",
    round(float(output.max()), 4)
)

Entrada: torch.Size([1, 1, 1024, 256])
Saída: torch.Size([1, 1, 1024, 256])
Tempo: 1.14 segundos
VRAM máxima: 0.12 GB
Máscara: 0.1581 até 0.9372


----

## 6. Função de perda com máscara de validade

O treinamento da rede de separação de bateria utiliza uma função de perda do tipo L1 com máscara de validade (`masked_l1_loss`). Essa função mede o erro absoluto médio entre a máscara espectral prevista pelo modelo e a máscara ideal (*ground truth*), considerando apenas as regiões do espectro definidas como válidas.

Sejam:

- \( \hat{M} \in \mathbb{R}^{B \times F \times T} \) a máscara prevista pela rede;
- \( M \in \mathbb{R}^{B \times F \times T} \) a máscara ideal;
- \( V \in \{0,1\}^{B \times F} \) a máscara binária de validade, definida no domínio da frequência,

em que:

- \( B \) é o tamanho do *batch*;
- \( F \) é o número de bandas de frequência (bins do espectro);
- \( T \) é o número de quadros de tempo (frames do espectro).

Primeiramente, calcula-se o erro absoluto por pixel:

\[
L_{\text{pixel}} = \left| \hat{M} - M \right|.
\]

A máscara de validade \( V \) é expandida ao longo da dimensão temporal para que tenha o mesmo formato de \( L_{\text{pixel}} \), resultando em \( \tilde{V} \in \{0,1\}^{B \times F \times T} \). Em seguida, a perda total é definida como:

\[
\mathcal{L} = \frac{\sum_{b,f,t} L_{\text{pixel}}[b,f,t] \cdot \tilde{V}[b,f,t]}{\sum_{b,f,t} \tilde{V}[b,f,t] + \varepsilon},
\]

onde \( \varepsilon = 10^{-8} \) é uma constante numérica para evitar divisão por zero.

Dessa forma, regiões do espectro marcadas como inválidas (\( V = 0 \)) não contribuem para o cálculo da perda, permitindo que o modelo seja treinado apenas nas bandas de frequência consideradas relevantes para a tarefa de separação de bateria.

In [17]:
def masked_l1_loss(
    prediction,
    target,
    valid_mask
):
    # Quanto mais próximo de zero, mais parecida com a máscara-alvo
    pixel_loss = torch.abs(
        prediction - target
    )

    valid_mask = valid_mask.unsqueeze(2)
    valid_mask = valid_mask.expand_as(
        pixel_loss
    )

    return (pixel_loss * valid_mask).sum() / (
        valid_mask.sum() + 1e-8
    )

----

## 7. Construção do conjunto de dados por segmentos

A classe `DrumChunkDataset` foi desenvolvida para preparar os dados utilizados no treinamento da rede de separação de bateria. Em vez de fornecer músicas completas ao modelo, cada faixa é dividida em segmentos menores, denominados *chunks*, contendo uma quantidade fixa de quadros temporais do espectrograma.

Para cada faixa, o dataset utiliza dois arquivos: `mixture.wav`, que contém a mistura completa de instrumentos, e `drums.wav`, que contém apenas a bateria. Inicialmente, a duração de cada arquivo é consultada e a faixa é dividida em segmentos consecutivos de `FRAMES_PER_CHUNK` quadros de STFT. Dessa forma, uma única música pode gerar diversos exemplos de treinamento.

Ao recuperar um exemplo, o dataset carrega o mesmo intervalo temporal da mistura e da faixa de bateria. Os dois sinais são carregados com a mesma taxa de amostragem, convertidos para mono e cortados para o menor comprimento disponível. Esse procedimento garante o alinhamento temporal entre o sinal de entrada e a referência usada como alvo.

Em seguida, os sinais são convertidos para o domínio tempo-frequência por meio da Transformada de Fourier de Tempo Curto (STFT). A magnitude do espectrograma da mistura é utilizada como entrada da rede. A magnitude da bateria é empregada para construir a máscara-alvo, definida como a razão entre a magnitude da bateria e a magnitude da mistura. Os valores da máscara são limitados ao intervalo entre 0 e 1.

A magnitude da mistura é convertida para decibéis e normalizada para o intervalo entre 0 e 1. Essa normalização reduz a variação numérica dos dados de entrada e fornece uma escala mais adequada ao treinamento da rede neural.

Os segmentos localizados no final de uma faixa podem possuir menos quadros temporais que o tamanho definido para um chunk. Nesses casos, a entrada e a máscara-alvo são preenchidas com zeros até atingirem o tamanho fixo. Paralelamente, é criada uma máscara de validade temporal: os frames originais recebem valor 1 e os frames adicionados por preenchimento recebem valor 0. Essa máscara é utilizada na função de perda para impedir que os valores artificiais de padding influenciem o treinamento.

Por fim, cada exemplo retorna três tensores: o espectrograma normalizado da mistura, a máscara-alvo da bateria e a máscara de validade temporal. Após o agrupamento pelo `DataLoader`, os tensores possuem as seguintes dimensões:

- `mixture`: batch × 1 × frequência × tempo;
- `target_mask`: batch × 1 × frequência × tempo;
- `valid_mask`: batch × 1 × tempo.

A dimensão de canal com tamanho igual a 1 é utilizada porque a U-Net recebe dados monofônicos organizados no formato esperado por camadas convolucionais bidimensionais.

In [18]:
class DrumChunkDataset(Dataset):
    def __init__(
        self,
        track_dirs,
        frames_per_chunk=FRAMES_PER_CHUNK,
        augment=False
    ):
        self.track_dirs = list(track_dirs)
        self.frames_per_chunk = frames_per_chunk
        self.augment = augment
        self.examples = []

        for track_dir in self.track_dirs:
            mixture_path = track_dir / "mixture.wav"
            drums_path = track_dir / "drums.wav"

            audio_info = sf.info(mixture_path)

            total_frames = max(
                1,
                1 + (
                    audio_info.frames - N_FFT
                ) // HOP_LENGTH
            )

            for start_frame in range(
                0,
                total_frames,
                self.frames_per_chunk
            ):
                self.examples.append(
                    (
                        mixture_path,
                        drums_path,
                        start_frame
                    )
                )

    def __len__(self):
        return len(self.examples)

    def __getitem__(self, index):
        mixture_path, drums_path, start_frame = (
            self.examples[index]
        )

        sample_start = start_frame * HOP_LENGTH

        audio_length = (
            self.frames_per_chunk * HOP_LENGTH
            + N_FFT
        )

        y_mix, _ = librosa.load(
            mixture_path,
            sr=SAMPLE_RATE,
            mono=True,
            offset=sample_start / SAMPLE_RATE,
            duration=audio_length / SAMPLE_RATE
        )

        y_drums, _ = librosa.load(
            drums_path,
            sr=SAMPLE_RATE,
            mono=True,
            offset=sample_start / SAMPLE_RATE,
            duration=audio_length / SAMPLE_RATE
        )

        # Cortar primeiro
        length = min(len(y_mix), len(y_drums))

        y_mix = y_mix[:length]
        y_drums = y_drums[:length]

        # Data augmentation (somente no treino)
        if self.augment:
            # Random gain
            gain = np.random.uniform(0.7, 1.3)
            y_mix = y_mix * gain
            y_drums = y_drums * gain
    
            # Ruído (somente na mistura)
            # if np.random.random() > 0.5:
            #     noise_level = np.random.uniform(0.0, 0.005)
            #     noise = np.random.randn(length) * noise_level
            #     y_mix = y_mix + noise

        y_mix = y_mix[:length]
        y_drums = y_drums[:length]

        mix_stft = librosa.stft(
            y_mix,
            n_fft=N_FFT,
            hop_length=HOP_LENGTH
        )

        drums_stft = librosa.stft(
            y_drums,
            n_fft=N_FFT,
            hop_length=HOP_LENGTH
        )


        mix_mag = np.abs(mix_stft)[:-1]
        drums_mag = np.abs(drums_stft)[:-1]
      

        target_mask = drums_mag / (mix_mag + 1e-8)
        target_mask = np.clip(target_mask, 0, 1)

        mix_db = librosa.amplitude_to_db(
            mix_mag,
            ref=np.max
        )

        mix_input = np.clip(
            (mix_db + 80.0) / 80.0,
            0.0,
            1.0
        )

        valid_frames = min(
            mix_input.shape[1],
            self.frames_per_chunk
        )

        mix_input = mix_input[
            :, :self.frames_per_chunk
        ]

        target_mask = target_mask[
            :, :self.frames_per_chunk
        ]

        if valid_frames < self.frames_per_chunk:
            pad = (
                self.frames_per_chunk
                - valid_frames
            )

            mix_input = np.pad(
                mix_input,
                ((0, 0), (0, pad)),
                mode="constant"
            )

            target_mask = np.pad(
                target_mask,
                ((0, 0), (0, pad)),
                mode="constant"
            )

        valid_mask = np.zeros(
            (1, self.frames_per_chunk),
            dtype=np.float32
        )

        valid_mask[
            :, :valid_frames
        ] = 1.0

        return {
            "mixture": torch.tensor(
                mix_input,
                dtype=torch.float32
            ).unsqueeze(0),

            "target_mask": torch.tensor(
                target_mask,
                dtype=torch.float32
            ).unsqueeze(0),

            "valid_mask": torch.tensor(
                valid_mask,
                dtype=torch.float32
            )
        }

### [TESTE_PIPELINE] Verificação do carregamento dos segmentos

Após a implementação da classe `DrumChunkDataset`, foi realizado um teste de funcionamento utilizando uma única faixa de áudio. O objetivo foi verificar a quantidade de segmentos gerados, o tempo necessário para carregar e pré-processar um chunk e os formatos dos tensores retornados pelo `DataLoader`.

O teste confirmou que cada faixa é dividida em múltiplos segmentos temporais e que, após o agrupamento em batch, os dados são estruturados nos formatos esperados pela rede: `batch × 1 × frequência × tempo` para a mistura e para a máscara-alvo, e `batch × 1 × tempo` para a máscara de validade.

Essa etapa teve caráter diagnóstico e foi utilizada para validar a compatibilidade entre o pré-processamento dos dados, o `DataLoader` e a arquitetura U-Net antes do início do treinamento.

In [19]:
one_real_dataset = DrumChunkDataset(
    [first_track]
)

one_real_loader = DataLoader(
    one_real_dataset,
    batch_size=1,
    shuffle=False,
    num_workers=0,
    pin_memory=True
)

print("Chunks da faixa:", len(one_real_dataset))

start = time.time()

batch = next(iter(one_real_loader))

print(
    "Tempo para carregar 1 chunk:",
    round(time.time() - start, 3),
    "segundos"
)

print("Mixture:", batch["mixture"].shape)
print("Target:", batch["target_mask"].shape)
print("Valid mask:", batch["valid_mask"].shape)

Chunks da faixa: 58
Tempo para carregar 1 chunk: 0.069 segundos
Mixture: torch.Size([1, 1, 1024, 256])
Target: torch.Size([1, 1, 1024, 256])
Valid mask: torch.Size([1, 1, 256])


---

## 8. Criação dos conjuntos de treinamento e teste

Após a preparação da classe `DrumChunkDataset`, foram criados os conjuntos de dados destinados ao treinamento e à avaliação do modelo. O conjunto de treinamento foi construído a partir das faixas presentes em `train_tracks`, enquanto o conjunto de teste foi formado pelas faixas presentes em `test_tracks`.

Cada faixa é dividida em múltiplos segmentos temporais de tamanho fixo. Portanto, o número de exemplos disponível em cada conjunto não corresponde diretamente ao número de músicas, mas ao total de chunks extraídos de todas as faixas. Cada chunk contém o espectrograma normalizado da mistura, a máscara-alvo da bateria e a máscara de validade temporal.

No experimento apresentado, o aumento de dados foi desativado para ambos os conjuntos, utilizando-se `augment=False`. Dessa forma, as faixas de treinamento foram processadas sem alteração artificial de ganho ou adição de ruído. No conjunto de teste, a ausência de aumento de dados é necessária para que a avaliação seja realizada sobre sinais originais e seja reproduzível entre diferentes execuções.

A quantidade de chunks de treinamento e teste foi exibida como etapa de verificação, permitindo confirmar a quantidade total de exemplos que seria disponibilizada ao processo de aprendizagem e à avaliação do modelo.

In [20]:
train_dataset = DrumChunkDataset(
    train_tracks,
    augment=False  #  Augmentation no treino
)


test_dataset = DrumChunkDataset(
    test_tracks,
)

print("Chunks de treino:", len(train_dataset))
print("Chunks de teste:", len(test_dataset))

Chunks de treino: 7753
Chunks de teste: 4219


---

## 9. Organização dos dados em batches

Após a criação dos conjuntos de treinamento e teste, foram utilizados objetos `DataLoader` do PyTorch para organizar o fornecimento dos exemplos ao modelo. Os `DataLoader`s são responsáveis por recuperar os chunks produzidos pelo dataset, agrupá-los em batches e disponibilizá-los ao loop de treinamento ou de avaliação.

Para o conjunto de treinamento, foi utilizado `batch_size=1`, indicando que cada atualização da rede é realizada a partir de um único chunk. A ordem dos exemplos foi embaralhada com `shuffle=True` a cada época. Esse procedimento reduz a dependência entre batches consecutivos, pois chunks próximos de uma mesma música podem apresentar características semelhantes.

Para o conjunto de teste, também foi utilizado `batch_size=1`, porém sem embaralhamento (`shuffle=False`). Como a avaliação não atualiza os pesos do modelo, manter uma ordem fixa facilita a reprodutibilidade dos resultados.

O parâmetro `num_workers=0` determina que o carregamento dos dados ocorra no processo principal de execução. Além disso, `pin_memory=True` foi utilizado para permitir transferências potencialmente mais eficientes dos tensores para a GPU, quando disponível.

Como o tamanho do batch foi definido como 1, o número de batches em cada conjunto é igual ao número total de chunks gerados pelo respectivo dataset.

In [21]:
train_loader = DataLoader(
    train_dataset,
    batch_size=1,
    shuffle=True,
    num_workers=0,
    pin_memory=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=1,
    shuffle=False,
    num_workers=0,
    pin_memory=True
)

print("Batches de treino:", len(train_loader))
print("Batches de teste:", len(test_loader))

Batches de treino: 7753
Batches de teste: 4219


In [22]:
model_unet = UNetAudio().to(device)

optimizer = optim.AdamW(
    model_unet.parameters(),
    lr=1e-3,
    weight_decay=1e-3
)

model_unet.train()

start = time.time()

for batch_idx, batch in enumerate(
    train_loader,
    start=1
):
    mixture = batch["mixture"].to(
        device,
        non_blocking=True
    )

    target_mask = batch["target_mask"].to(
        device,
        non_blocking=True
    )

    valid_mask = batch["valid_mask"].to(
        device,
        non_blocking=True
    )

    optimizer.zero_grad(set_to_none=True)

    predicted_mask = model_unet(mixture)

    loss = masked_l1_loss(
        predicted_mask,
        target_mask,
        valid_mask
    )

    loss.backward()
    optimizer.step()

    if batch_idx % 25 == 0:
        print(
            f"Batch {batch_idx}/100 | "
            f"Loss: {loss.item():.6f}"
        )

    if batch_idx == 100:
        break

torch.cuda.synchronize()

elapsed = time.time() - start

print(f"100 batches em {elapsed:.2f} s")
print(f"Média: {elapsed / 100:.3f} s/batch")

Batch 25/100 | Loss: 0.407654
Batch 50/100 | Loss: 0.350797
Batch 75/100 | Loss: 0.332036
Batch 100/100 | Loss: 0.357577
100 batches em 12.42 s
Média: 0.124 s/batch


---

## 10. Inicialização do modelo e configuração da otimização

A rede foi inicializada por meio da classe `UNetAudio` e transferida para o dispositivo de processamento disponível. A inicialização da arquitetura estabelece novos pesos aleatórios, garantindo que o treinamento seja iniciado sem reutilização involuntária de parâmetros de execuções anteriores.

A atualização dos parâmetros foi realizada pelo otimizador AdamW. Esse método utiliza os gradientes calculados por retropropagação para ajustar os pesos da rede após cada batch. A taxa de aprendizado inicial foi definida como 0,00005, controlando a intensidade das atualizações realizadas durante o treinamento. Também foi utilizado decaimento de pesos (*weight decay*) de 0,001 como mecanismo de regularização, com o objetivo de reduzir a tendência de sobreajuste.

Além do otimizador, foi utilizado o agendador `ReduceLROnPlateau` para ajustar dinamicamente a taxa de aprendizado de acordo com o desempenho de validação. O agendador monitora a função de perda de validação e reduz a taxa de aprendizado pela metade quando não observa melhora significativa durante duas épocas consecutivas. A taxa mínima foi limitada a 0,000001.

Essa configuração permite que o treinamento comece com atualizações mais amplas e, caso a redução da perda de validação se estabilize, passe a utilizar atualizações menores, favorecendo um ajuste mais refinado dos parâmetros da rede.

In [23]:
model_unet = UNetAudio().to(device)

optimizer = optim.AdamW(
    model_unet.parameters(),
    lr=5e-5,
    weight_decay=1e-3
)

scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode="min",
    factor=0.5,
    patience=2,
    threshold=1e-3,
    min_lr=1e-6
)

print("Modelo e otimizador reiniciados.")

Modelo e otimizador reiniciados.


----

## 11. Avaliação por SI-SDR

A qualidade da separação de bateria foi avaliada por meio da métrica SI-SDR (*Scale-Invariant Signal-to-Distortion Ratio*). Essa métrica compara o sinal de bateria estimado pelo modelo com a faixa de bateria real e mede a relação entre a parcela da estimativa que corresponde ao sinal desejado e a parcela correspondente a ruídos, distorções ou vazamento de outros instrumentos.

O termo *scale-invariant* indica que a métrica é invariável a diferenças globais de amplitude. Dessa forma, uma estimativa com conteúdo semelhante ao sinal de referência, mas com volume mais alto ou mais baixo, não é penalizada apenas pela diferença de ganho. Antes do cálculo, os sinais de referência e estimado são convertidos para vetores, cortados para o mesmo comprimento e centralizados pela remoção de suas médias.

Para calcular a métrica, o sinal estimado é projetado sobre o sinal de referência. A parcela projetada representa o componente desejado, enquanto a diferença entre a estimativa e essa projeção corresponde ao componente de ruído ou distorção. O SI-SDR é calculado pela razão, em decibéis, entre a energia do componente desejado e a energia do componente de ruído. Valores maiores indicam uma estimativa mais próxima da bateria real.

Foi também calculada uma baseline utilizando a mistura completa como uma estimativa simplificada da bateria. Nessa baseline, o arquivo `mixture.wav` é comparado diretamente ao arquivo `drums.wav`, sem aplicação de qualquer método de separação. O desempenho do modelo pode, então, ser comparado a esse valor de referência. Quando o SI-SDR estimado pelo modelo é superior ao da baseline, isso indica que a rede reduziu parte da interferência dos demais instrumentos presentes na mistura.

Para avaliar uma faixa completa, a mistura é carregada e convertida para o domínio tempo-frequência por meio da STFT. A magnitude do espectrograma é convertida para decibéis e normalizada utilizando o mesmo procedimento adotado durante o treinamento. Em seguida, o espectrograma é dividido em chunks temporais com tamanho fixo. Cada chunk é fornecido à U-Net, que estima uma máscara espectral para a bateria.

A máscara prevista é limitada ao intervalo entre 0 e 1 e multiplicada pela magnitude original da mistura. O resultado é uma estimativa da magnitude espectral da bateria. Para reconstruir o sinal no domínio do tempo, essa magnitude estimada é combinada com a fase da mistura original e processada pela Transformada Inversa de Fourier de Tempo Curto (ISTFT).

A avaliação é realizada individualmente para cada faixa de validação ou teste. Após a obtenção do SI-SDR de cada música, calcula-se a média aritmética dos valores válidos. Dessa forma, cada faixa possui o mesmo peso no resultado final, independentemente de sua duração.

In [24]:
import numpy as np
import librosa
import torch
from pathlib import Path

def compute_si_sdr(s_ref, s_est):
    s_ref = np.asarray(
        s_ref,
        dtype=np.float64
    ).reshape(-1)

    s_est = np.asarray(
        s_est,
        dtype=np.float64
    ).reshape(-1)

    length = min(
        len(s_ref),
        len(s_est)
    )

    s_ref = s_ref[:length]
    s_est = s_est[:length]

    s_ref = s_ref - np.mean(s_ref)
    s_est = s_est - np.mean(s_est)

    eps = 1e-12

    ref_energy = np.sum(s_ref ** 2)

    if ref_energy < eps:
        return np.nan

    scale = np.dot(
        s_est,
        s_ref
    ) / ref_energy

    target = scale * s_ref
    noise = s_est - target

    target_energy = np.sum(target ** 2)
    noise_energy = np.sum(noise ** 2)

    if noise_energy < eps:
        return float("inf")

    return float(
        10.0 * np.log10(
            target_energy / noise_energy
        )
    )


def compute_baseline_si_sdr(
    mix_path,
    drums_path,
    sample_rate=44100
):
    """
    Calcula SI-SDR entre a mistura e a bateria real.
    Isso serve como baseline: se o modelo for melhor que isso,
    está removendo interferências.
    """
    y_mix, _ = librosa.load(
        mix_path,
        sr=sample_rate,
        mono=True
    )

    y_drums, _ = librosa.load(
        drums_path,
        sr=sample_rate,
        mono=True
    )

    length = min(len(y_mix), len(y_drums))

    return compute_si_sdr(
        y_drums[:length],
        y_mix[:length]
    )
@torch.no_grad()
def separate_track_for_si_sdr(
    model,
    mix_path,
    device,
    sample_rate=44100,
    n_fft=2048,
    hop_length=512,
    frames_per_chunk=256
):
    """
    Estima a bateria de uma faixa inteira usando exatamente o mesmo
    pré-processamento do DrumChunkDataset.
    """

    model.eval()

    y_mix, _ = librosa.load(
        mix_path,
        sr=sample_rate,
        mono=True
    )

    mix_stft = librosa.stft(
        y_mix,
        n_fft=n_fft,
        hop_length=hop_length
    )

    # Mesma transformação do dataset: remover último bin
    mix_mag = np.abs(mix_stft)[:-1]          # [1024, time]
    mix_phase = np.angle(mix_stft)[:-1]      # [1024, time]

    mix_db = librosa.amplitude_to_db(
        mix_mag,
        ref=np.max

    )

    mix_input = np.clip(
        (mix_db + 80.0) / 80.0,
        0.0,
        1.0
    ).astype(np.float32)

    n_freq, n_time = mix_input.shape
    n_chunks = int(np.ceil(n_time / frames_per_chunk))

    estimated_mag = np.zeros_like(mix_mag, dtype=np.float32)
    estimated_weight = np.zeros_like(mix_mag, dtype=np.float32)

    for chunk_index in range(n_chunks):
        t0 = chunk_index * frames_per_chunk
        t1 = min(t0 + frames_per_chunk, n_time)

        valid_frames = t1 - t0

        chunk = mix_input[:, t0:t1]

        if valid_frames < frames_per_chunk:
            pad_width = frames_per_chunk - valid_frames

            chunk = np.pad(
                chunk,
                ((0, 0), (0, pad_width)),
                mode="constant"
            )

        chunk_tensor = torch.from_numpy(chunk)
        chunk_tensor = chunk_tensor.unsqueeze(0).unsqueeze(0)
        chunk_tensor = chunk_tensor.to(device)

        predicted_mask = model(chunk_tensor)
        predicted_mask = predicted_mask.squeeze().cpu().numpy()

        predicted_mask = np.clip(
            predicted_mask,
            0.0,
            1.0
        )

        predicted_mask = predicted_mask[:, :valid_frames]

        estimated_mag[:, t0:t1] += (
            mix_mag[:, t0:t1] * predicted_mask
        )

        estimated_weight[:, t0:t1] += 1.0

    estimated_mag = estimated_mag / np.maximum(
        estimated_weight,
        1e-8
    )

    # Reconstruir STFT com 1025 bins (adicionar último bin zerado)
    estimated_mag_full = np.concatenate(
        [
            estimated_mag,
            np.zeros(
                (1, estimated_mag.shape[1]),
                dtype=estimated_mag.dtype
            )
        ],
        axis=0
    )

    mix_phase_full = np.concatenate(
        [
            mix_phase,
            np.zeros(
                (1, mix_phase.shape[1]),
                dtype=mix_phase.dtype
            )
        ],
        axis=0
    )

    estimated_stft = (
        estimated_mag_full *
        np.exp(1j * mix_phase_full)
    )

    estimated_audio = librosa.istft(
        estimated_stft,
        hop_length=hop_length,
        length=len(y_mix)
    )

    return estimated_audio

def calculate_baseline_mean(
    tracks,
    sample_rate=44100
):
    """Calcula SI-SDR médio da baseline (mistura vs bateria) em várias faixas."""
    
    values = []
    
    for track_dir in tracks:
        mix_path = Path(track_dir) / "mixture.wav"
        drums_path = Path(track_dir) / "drums.wav"
        
        if not mix_path.exists() or not drums_path.exists():
            continue
        
        value = compute_baseline_si_sdr(
            mix_path,
            drums_path,
            sample_rate
        )
        
        if np.isfinite(value):
            values.append(value)
    
    if not values:
        return float("nan")
    
    return float(np.mean(values))
    
@torch.no_grad()
def calculate_validation_si_sdr(
    model,
    val_tracks,
    device,
    sample_rate=44100,
    n_fft=2048,
    hop_length=512,
    frames_per_chunk=256
):
    """
    Calcula SI-SDR médio usando as faixas completas de validação/teste.
    Cada faixa precisa conter mixture.wav e drums.wav.
    """

    model.eval()
    values = []

    for track_dir in val_tracks:
        mix_path = Path(track_dir) / "mixture.wav"
        drums_path = Path(track_dir) / "drums.wav"

        if not mix_path.exists() or not drums_path.exists():
            continue

        y_drums, _ = librosa.load(
            drums_path,
            sr=sample_rate,
            mono=True
        )

        y_estimated = separate_track_for_si_sdr(
            model=model,
            mix_path=mix_path,
            device=device,
            sample_rate=sample_rate,
            n_fft=n_fft,
            hop_length=hop_length,
            frames_per_chunk=frames_per_chunk
        )

        length = min(
            len(y_drums),
            len(y_estimated)
        )

        si_sdr = compute_si_sdr(
            y_drums[:length],
            y_estimated[:length]
        )

        if np.isfinite(si_sdr):
            values.append(si_sdr)

    if not values:
        return float("nan")

    return float(np.mean(values))

In [25]:
# Criar uma instância apenas para inspecionar a arquitetura
model_summary = UNetAudio().to(device)

total_params = sum(
    parameter.numel()
    for parameter in model_summary.parameters()
)

trainable_params = sum(
    parameter.numel()
    for parameter in model_summary.parameters()
    if parameter.requires_grad
)

non_trainable_params = total_params - trainable_params

print(f"Parâmetros totais: {total_params:,}")
print(f"Parâmetros treináveis: {trainable_params:,}")
print(f"Parâmetros não treináveis: {non_trainable_params:,}")

# Estimativa de memória dos pesos em float32
model_size_mb = total_params * 4 / (1024 ** 2)

print(f"Tamanho aproximado dos pesos: {model_size_mb:.2f} MB")

Parâmetros totais: 1,962,337
Parâmetros treináveis: 1,962,337
Parâmetros não treináveis: 0
Tamanho aproximado dos pesos: 7.49 MB


---

## 12. Treinamento, validação e armazenamento de checkpoints

O treinamento foi realizado durante 10 épocas. Em cada época, todos os batches do conjunto de treinamento foram processados pela U-Net. Para cada batch, o espectrograma normalizado da mistura foi fornecido ao modelo, que produziu uma máscara espectral estimada para a bateria. Essa máscara foi comparada à máscara-alvo utilizando a função de perda L1 com máscara de validade.

Após o cálculo da perda, os gradientes foram obtidos por retropropagação e os parâmetros da rede foram atualizados pelo otimizador AdamW. Antes da atualização dos pesos, foi aplicado o recorte de gradientes (*gradient clipping*), limitando a norma dos gradientes a 1,0. Esse procedimento foi utilizado para reduzir a possibilidade de atualizações excessivas e instabilidades durante a otimização.

A loss de treinamento de cada época foi obtida pela média das losses calculadas em todos os batches do conjunto de treinamento. A avaliação da loss foi realizada com o modelo no modo de avaliação, sem cálculo de gradientes. Para reduzir o custo computacional, a loss de validação foi estimada utilizando no máximo 500 batches do conjunto de teste.

Além da loss, foi utilizada a métrica SI-SDR para avaliar a qualidade do áudio separado. A avaliação foi realizada em 50 faixas completas do conjunto de teste. Para cada faixa, o modelo estimou uma máscara de bateria, aplicou essa máscara à magnitude do espectrograma da mistura e reconstruiu o sinal de áudio por meio da ISTFT. O sinal estimado foi então comparado à faixa de bateria real.

Como referência, foi calculada uma baseline de SI-SDR utilizando a própria mistura como estimativa de bateria, sem qualquer processo de separação. A baseline e o modelo foram avaliados nas mesmas 50 faixas, permitindo uma comparação direta entre o desempenho sem separação e o desempenho obtido pela U-Net. A diferença entre o SI-SDR do modelo e a baseline representa a melhoria atribuída ao processo de separação.

A taxa de aprendizado foi ajustada dinamicamente por meio do agendador `ReduceLROnPlateau`, que monitora a loss de validação. Quando não era observada melhora significativa durante o número definido de épocas, a taxa de aprendizado era reduzida pela metade, respeitando um valor mínimo previamente estabelecido.

Ao final de cada época, foram armazenados checkpoints contendo os pesos da rede, os estados do otimizador e do agendador, as métricas da época e o histórico acumulado. Três arquivos foram mantidos: o checkpoint da última época concluída (`last.pt`), o modelo com menor loss de validação (`best.pt`) e o modelo com maior SI-SDR (`best_si_sdr.pt`). O checkpoint selecionado para a separação final foi o associado ao maior SI-SDR, por estar diretamente relacionado à qualidade do áudio reconstruído.

In [65]:
EPOCHS = 30
MAX_VAL_BATCHES = 500

best_val_loss = float("inf")
best_val_si_sdr = -float("inf")

history = {
    "train_loss": [],
    "val_loss": [],
    "val_si_sdr": []
}

# Definir faixas de avaliação (usadas para baseline E SI-SDR)
eval_tracks = test_tracks[:50]

# Calcular baseline média nessas mesmas faixas
baseline_si_sdr = calculate_baseline_mean(
    eval_tracks,
    SAMPLE_RATE
)

print(
    f"Baseline SI-SDR "
    f"(média em {len(eval_tracks)} faixas): "
    f"{baseline_si_sdr:.2f} dB"
)

for epoch in range(1, EPOCHS + 1):

    
    # Treinamento
    model_unet.train()

    train_loss_total = 0.0

    for batch_idx, batch in enumerate(
        train_loader,
        start=1
    ):
        mixture = batch["mixture"].to(
            device,
            non_blocking=True
        )

        target_mask = batch["target_mask"].to(
            device,
            non_blocking=True
        )

        valid_mask = batch["valid_mask"].to(
            device,
            non_blocking=True
        )

        optimizer.zero_grad(
            set_to_none=True
        )

        predicted_mask = model_unet(
            mixture
        )

        loss = masked_l1_loss(
            predicted_mask,
            target_mask,
            valid_mask
        )
        
        loss.backward()

        torch.nn.utils.clip_grad_norm_(
            model_unet.parameters(),
            max_norm=1.0
        )

        optimizer.step()

        train_loss_total += loss.item()

        if batch_idx % 500 == 0:
            print(
                f"Época {epoch}/{EPOCHS} | "
                f"Batch {batch_idx}/{len(train_loader)} | "
                f"Loss: {loss.item():.6f}"
            )

    train_loss = (
        train_loss_total /
        len(train_loader)
    )

    # Validação da loss
    model_unet.eval()

    val_loss_total = 0.0
    val_batches_used = 0

    with torch.no_grad():
        for batch in test_loader:

            mixture = batch["mixture"].to(
                device,
                non_blocking=True
            )

            target_mask = batch["target_mask"].to(
                device,
                non_blocking=True
            )

            valid_mask = batch["valid_mask"].to(
                device,
                non_blocking=True
            )

            predicted_mask = model_unet(
                mixture
            )

            loss = masked_l1_loss(
                predicted_mask,
                target_mask,
                valid_mask
            )

            val_loss_total += loss.item()
            val_batches_used += 1

            if val_batches_used >= MAX_VAL_BATCHES:
                break

    val_loss = (
        val_loss_total /
        max(val_batches_used, 1)
    )

    # SI-SDR
    val_si_sdr = calculate_validation_si_sdr(
        model=model_unet,
         val_tracks=eval_tracks,  # ← Usa as mesmas faixas da baseline
        device=device,
        sample_rate=SAMPLE_RATE,
          n_fft=N_FFT,
        hop_length=HOP_LENGTH,
        frames_per_chunk=FRAMES_PER_CHUNK
    )


    # Scheduler
    scheduler.step(val_loss)

    current_lr = optimizer.param_groups[0]["lr"]

    # Histórico
    history["train_loss"].append(train_loss)
    history["val_loss"].append(val_loss)
    history["val_si_sdr"].append(val_si_sdr)

    print()
    print(
        f"Época {epoch}/{EPOCHS} concluída | "
        f"Treino: {train_loss:.6f} | "
        f"Validação: {val_loss:.6f} | "
        f"SI-SDR: {val_si_sdr:.2f} dB | "
        f"Baseline: {baseline_si_sdr:.2f} dB | "
        f"LR: {current_lr:.2e}"
    )

    # Checkpoint
    checkpoint = {
        "epoch": epoch,
        "model_state_dict": model_unet.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "scheduler_state_dict": scheduler.state_dict(),
        "train_loss": train_loss,
        "val_loss": val_loss,
        "val_si_sdr": val_si_sdr,
        "history": history
    }

    torch.save(
        checkpoint,
        CHECKPOINT_DIR / "last.pt"
    )

    # Salvar melhor modelo por loss
    if val_loss < best_val_loss:
        best_val_loss = val_loss

        torch.save(
            checkpoint,
            CHECKPOINT_DIR / "best.pt"
        )

        print("✓ Melhor loss salva.")

    # Salvar melhor modelo por SI-SDR
    if val_si_sdr > best_val_si_sdr:
        best_val_si_sdr = val_si_sdr

        torch.save(
            checkpoint,
            CHECKPOINT_DIR / "best_si_sdr.pt"
        )

        print("✓ Melhor SI-SDR salvo.")

    print()

Baseline SI-SDR (média em 50 faixas): -3.69 dB
Época 1/30 | Batch 500/7753 | Loss: 0.148754
Época 1/30 | Batch 1000/7753 | Loss: 0.224185
Época 1/30 | Batch 1500/7753 | Loss: 0.187636
Época 1/30 | Batch 2000/7753 | Loss: 0.240199
Época 1/30 | Batch 2500/7753 | Loss: 0.173353
Época 1/30 | Batch 3000/7753 | Loss: 0.092225
Época 1/30 | Batch 3500/7753 | Loss: 0.155395
Época 1/30 | Batch 4000/7753 | Loss: 0.203348
Época 1/30 | Batch 4500/7753 | Loss: 0.092696
Época 1/30 | Batch 5000/7753 | Loss: 0.201789
Época 1/30 | Batch 5500/7753 | Loss: 0.225230
Época 1/30 | Batch 6000/7753 | Loss: 0.118765
Época 1/30 | Batch 6500/7753 | Loss: 0.292217
Época 1/30 | Batch 7000/7753 | Loss: 0.228447
Época 1/30 | Batch 7500/7753 | Loss: 0.279227

Época 1/30 concluída | Treino: 0.183402 | Validação: 0.217214 | SI-SDR: -4.80 dB | Baseline: -3.69 dB | LR: 5.00e-05
✓ Melhor loss salva.
✓ Melhor SI-SDR salvo.

Época 2/30 | Batch 500/7753 | Loss: 0.132588
Época 2/30 | Batch 1000/7753 | Loss: 0.092709
Época 2/30

## [INPUT] Inferência e geração do áudio separado

Após o treinamento, foi realizada a etapa de inferência para estimar a bateria presente em uma música externa ao processo de aprendizagem. Inicialmente, foi criada uma nova instância da arquitetura `UNetAudio` e carregado um checkpoint contendo os pesos treinados. O modelo foi colocado em modo de avaliação por meio de `model.eval()`, pois nessa etapa não ocorre cálculo de gradientes nem atualização de parâmetros.

A música de entrada foi carregada em formato monofônico e reamostrada para 44.100 Hz, mantendo a mesma taxa de amostragem utilizada durante o treinamento. Em seguida, o áudio foi convertido para o domínio tempo-frequência por meio da Transformada de Fourier de Curto Tempo (STFT).

Foram extraídas a magnitude e a fase do espectrograma da mistura. A magnitude foi convertida para decibéis, normalizada para o intervalo entre 0 e 1 e dividida em segmentos de 256 frames temporais. Esse pré-processamento foi mantido igual ao utilizado no `DrumChunkDataset`, garantindo que a U-Net recebesse dados na mesma representação aprendida durante o treinamento.

Cada segmento foi fornecido à rede, que estimou uma máscara espectral com valores limitados entre 0 e 1. A máscara prevista foi multiplicada pela magnitude original da mistura, produzindo uma estimativa da magnitude espectral da bateria. Nos casos em que o último segmento possuía menos de 256 frames, foi aplicado preenchimento com zeros apenas para compatibilizar o formato de entrada da rede; os frames artificiais não foram incorporados ao resultado final.

Para reconstruir o áudio, a magnitude estimada foi combinada com a fase original da mistura. Em seguida, foi aplicada a Transformada Inversa de Fourier de Curto Tempo (ISTFT), convertendo novamente o espectrograma estimado para o domínio temporal. O sinal resultante foi armazenado no arquivo `musica_drums.wav`, contendo a estimativa de bateria produzida pela U-Net.

In [3]:
from pathlib import Path
import numpy as np
import librosa
import soundfile as sf
import torch


# Configurações (mesmas do treino)
SAMPLE_RATE = 44100
N_FFT = 2048
HOP_LENGTH = 512
FRAMES_PER_CHUNK = 256

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Dispositivo:", device)


Dispositivo: cuda


In [31]:

# Carregar modelo
model = UNetAudio().to(device)

checkpoint_path = Path(r"C:/Users/phili/Documents/IA-II/checkpoints/best_si_sdr.pt")
audio_path = Path(r"C:/Users/phili/Documents/IA-II/musica.wav")

print("Checkpoint:", checkpoint_path, "-> existe:", checkpoint_path.exists())
print("Áudio:", audio_path, "-> existe:", audio_path.exists())

checkpoint = torch.load(
    checkpoint_path,
    map_location=device,
    weights_only=True
)

model.load_state_dict(checkpoint["model_state_dict"])
model.eval()
print("Modelo carregado com sucesso.")


# Carregar seu áudio
y, sr = librosa.load(
    audio_path,
    sr=SAMPLE_RATE,
    mono=True
)
print("Áudio carregado:", y.shape, "amostras, SR:", sr)



# Preparar STFT (igual ao dataset de treino)
mix_stft = librosa.stft(
    y,
    n_fft=N_FFT,
    hop_length=HOP_LENGTH
)

# Remover último bin (igual ao DrumChunkDataset)
mix_mag = np.abs(mix_stft)[:-1]
mix_phase = np.angle(mix_stft)[:-1]

# Normalização em dB (igual ao dataset de treino)
mix_db = librosa.amplitude_to_db(
    mix_mag,
    ref=np.max
)

mix_input = np.clip(
    (mix_db + 80.0) / 80.0,
    0.0,
    1.0
).astype(np.float32)


# Processar em chunks
n_freq, n_time = mix_input.shape
n_chunks = int(np.ceil(n_time / FRAMES_PER_CHUNK))

estimated_mag = np.zeros_like(mix_mag, dtype=np.float32)
estimated_weight = np.zeros_like(mix_mag, dtype=np.float32)

for i in range(n_chunks):
    t0 = i * FRAMES_PER_CHUNK
    t1 = min(t0 + FRAMES_PER_CHUNK, n_time)

    valid_frames = t1 - t0

    chunk = mix_input[:, t0:t1]

    if valid_frames < FRAMES_PER_CHUNK:
        pad_width = FRAMES_PER_CHUNK - valid_frames

        chunk = np.pad(
            chunk,
            ((0, 0), (0, pad_width)),
            mode="constant"
        )

    chunk_tensor = torch.from_numpy(chunk).float()
    chunk_tensor = chunk_tensor.unsqueeze(0).unsqueeze(0)
    chunk_tensor = chunk_tensor.to(device)

    with torch.no_grad():
        mask_pred = model(chunk_tensor)

    mask_pred = mask_pred.squeeze().cpu().numpy()
    mask_pred = np.clip(mask_pred, 0.0, 1.0)
    mask_pred = mask_pred[:, :valid_frames]

    estimated_mag[:, t0:t1] += mix_mag[:, t0:t1] * mask_pred
    estimated_weight[:, t0:t1] += 1.0

# Normalizar pela sobreposição
estimated_mag = estimated_mag / np.maximum(estimated_weight, 1e-8)

# Reconstruir STFT com 1025 bins (adicionar último bin zerado)
estimated_mag_full = np.concatenate(
    [
        estimated_mag,
        np.zeros(
            (1, estimated_mag.shape[1]),
            dtype=estimated_mag.dtype
        )
    ],
    axis=0
)

mix_phase_full = np.concatenate(
    [
        mix_phase,
        np.zeros(
            (1, mix_phase.shape[1]),
            dtype=mix_phase.dtype
        )
    ],
    axis=0
)

estimated_stft = estimated_mag_full * np.exp(1j * mix_phase_full)

# iSTFT
drums_rec = librosa.istft(
    estimated_stft,
    hop_length=HOP_LENGTH,
    length=len(y)
)

# Salvar resultado
out_path = Path(r"C:\Users\phili\Documents\IA-II\musica_drums.wav")
sf.write(out_path, drums_rec, SAMPLE_RATE)

print("Áudio de bateria estimado salvo em:", out_path)



Checkpoint: C:\Users\phili\Documents\IA-II\checkpoints\best_si_sdr.pt -> existe: True
Áudio: C:\Users\phili\Documents\IA-II\musica.wav -> existe: True
Modelo carregado com sucesso.
Áudio carregado: (13331969,) amostras, SR: 44100
Áudio de bateria estimado salvo em: C:\Users\phili\Documents\IA-II\musica_drums.wav
